# ML Network Intrusion Detector
**IBM SkillsBuild Cyber Security BootCamp 2026**

**Student:** Vanshika Maru  
**Email:** vanshikamaru0205@gmail.com

This educational notebook trains a Random Forest model to classify labeled network-flow rows as Normal (`BENIGN`) or Attack. It does not monitor live traffic or block attacks.

## 1. Setup and dataset
Install the packages with `pip install -r requirements.txt`. Download a labeled CSV from the [CIC-IDS2017 dataset page](https://www.unb.ca/cic/datasets/ids-2017.html). Place it in `data/` or change `DATA_PATH` below.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
DATA_PATH = Path('data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
MAX_ROWS = 100_000  # Set to None to load all rows
print('Dataset path:', DATA_PATH.resolve())

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found: {DATA_PATH}. Download a labeled CIC-IDS2017 CSV and update DATA_PATH.'
    )

df = pd.read_csv(DATA_PATH, nrows=MAX_ROWS, low_memory=False)
df.columns = df.columns.astype(str).str.strip()
label_matches = [c for c in df.columns if c.strip().lower() == 'label']
if not label_matches:
    raise ValueError("No 'Label' column was found. Use a labeled CIC-IDS2017 CSV.")
label_col = label_matches[0]
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
df = df.dropna(subset=[label_col]).copy()
labels = df[label_col].astype(str).str.strip().str.upper()
y = pd.Series(np.where(labels.eq('BENIGN'), 'Normal', 'Attack'), index=df.index, name='target')
X = df.drop(columns=[label_col]).copy()

# Remove identifier-like columns and constant columns to reduce leakage/noise.
drop_cols = []
for col in X.columns:
    low = col.strip().lower()
    if low in {'flow id', 'source ip', 'destination ip', 'timestamp'} or X[col].nunique(dropna=True) <= 1:
        drop_cols.append(col)
X = X.drop(columns=drop_cols, errors='ignore')

# Convert string columns to numeric when most non-missing values parse as numbers.
for col in X.select_dtypes(include=['object']).columns:
    converted = pd.to_numeric(X[col], errors='coerce')
    if X[col].notna().sum() and converted.notna().sum() / X[col].notna().sum() >= 0.90:
        X[col] = converted
X = X.dropna(axis=1, how='all')
print('Cleaned data shape:', X.shape)
print('Class counts:\n', y.value_counts())
print('Removed columns:', len(drop_cols))

## 2. Explore class balance
The chart below summarizes the binary target. A class imbalance is possible, so the model uses class weights and we review precision, recall, and F1-score as well as accuracy.

In [ ]:
y.value_counts().reindex(['Normal', 'Attack']).plot(kind='bar', title='Network Flow Labels', xlabel='Class', ylabel='Rows')
plt.tight_layout()
plt.show()

## 3. Train the classifier
The dataset is split into stratified training and testing sets. Numeric columns are median-imputed; categorical columns are imputed and one-hot encoded.

In [ ]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
transformers = []
if numeric_cols:
    numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    transformers.append(('numeric', numeric_pipe, numeric_cols))
if categorical_cols:
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    transformers.append(('categorical', categorical_pipe, categorical_cols))
if not transformers:
    raise ValueError('No usable feature columns remained after cleaning.')
preprocessor = ColumnTransformer(transformers=transformers)
model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE,
                                          class_weight='balanced', n_jobs=-1))
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

## 4. Evaluate and save results
Metrics below come from the actual test split. Do not copy example or estimated values into the report; use the values printed after your successful run.

In [ ]:
metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'macro_precision': float(precision_score(y_test, y_pred, average='macro', zero_division=0)),
    'macro_recall': float(recall_score(y_test, y_pred, average='macro', zero_division=0)),
    'macro_f1': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'features_used': int(X.shape[1]),
}
print(json.dumps(metrics, indent=2))
print('\nClassification report:\n')
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=['Normal', 'Attack'], cmap='Blues')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
joblib.dump(model, out_dir / 'network_intrusion_random_forest.joblib')
with open(out_dir / 'evaluation_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Saved model and metrics to:', out_dir.resolve())

## 5. Limitations
This is a learning prototype, not a production security control. Random splitting may overstate performance when similar flows occur in both sets. Performance can change on different networks, newer attack types, or differently prepared data. For stronger validation, use time- or capture-based holdouts and evaluate false negatives carefully.

## References
- [CIC-IDS2017 — Canadian Institute for Cybersecurity](https://www.unb.ca/cic/datasets/ids-2017.html)
- [scikit-learn documentation](https://scikit-learn.org/stable/)
- [pandas documentation](https://pandas.pydata.org/docs/)